In [ ]:

def format_match_events(events_response: Response) -> str:
    """
    Format the match events response into a readable string.
    
    Args:
        events_response: Response from get_match_events function
        
    Returns:
        Formatted string of the match events
    """
    if not isinstance(events_response, ValidResponse):
        return f"Error: {events_response.error}" # type: ignore
    
    data = events_response.data
    if "response" not in data or not data["response"]:
        return "No events data available"
    
    events = data["response"]
    
    # Group events by type for better readability
    goals = []
    cards = []
    substitutions = []
    var_events = []
    other_events = []
    
    for event in events:
        event_type = event.get("type", "").lower()
        if event_type == "goal":
            goals.append(event)
        elif event_type == "card":
            cards.append(event)
        elif event_type == "subst":
            substitutions.append(event)
        elif event_type == "var":
            var_events.append(event)
        else:
            other_events.append(event)
    
    # Format the output
    result = []
    
    if goals:
        result.append("⚽ GOALS:")
        for goal in goals:
            time = goal.get("time", {}).get("elapsed", "?")
            team = goal.get("team", {}).get("name", "Unknown Team")
            player = goal.get("player", {}).get("name", "Unknown Player")
            detail = goal.get("detail", "Goal")
            assist = goal.get("assist", {})
            assist_name = assist.get("name") if assist and assist.get("name") else "No assist"
            result.append(f"  {time}' - {team}: {player} ({detail}) - Assist: {assist_name}")
        result.append("")
    
    if cards:
        result.append("🟨🟥 CARDS:")
        for card in cards:
            time = card.get("time", {}).get("elapsed", "?")
            team = card.get("team", {}).get("name", "Unknown Team")
            player = card.get("player", {}).get("name", "Unknown Player")
            detail = card.get("detail", "Card")
            card_emoji = "🟥" if "Red" in detail else "🟨"
            result.append(f"  {time}' - {team}: {player} ({detail}) {card_emoji}")
        result.append("")
    
    if substitutions:
        result.append("🔄 SUBSTITUTIONS:")
        for sub in substitutions:
            time = sub.get("time", {}).get("elapsed", "?")
            team = sub.get("team", {}).get("name", "Unknown Team")
            player_out = sub.get("player", {}).get("name", "Unknown Player")
            assist = sub.get("assist", {})
            player_in = assist.get("name") if assist and assist.get("name") else "Unknown Player"
            detail = sub.get("detail", "Substitution")
            result.append(f"  {time}' - {team}: {player_out} ➡️ {player_in} ({detail})")
        result.append("")
    
    if var_events:
        result.append("📺 VAR EVENTS:")
        for var in var_events:
            time = var.get("time", {}).get("elapsed", "?")
            team = var.get("team", {}).get("name", "Unknown Team")
            player = var.get("player", {}).get("name", "Unknown Player")
            detail = var.get("detail", "VAR")
            result.append(f"  {time}' - {team}: {player} ({detail})")
        result.append("")
    
    if other_events:
        result.append("ℹ️ OTHER EVENTS:")
        for event in other_events:
            time = event.get("time", {}).get("elapsed", "?")
            team = event.get("team", {}).get("name", "Unknown Team")
            player = event.get("player", {}).get("name", "Unknown Player")
            event_type = event.get("type", "Unknown")
            detail = event.get("detail", "")
            result.append(f"  {time}' - {team}: {player} ({event_type} - {detail})")
    
    return "\n".join(result) if result else "No events found"